# Swap valuation and par coupon

Value a five-year receive-fixed USD swap and solve the coupon that makes its
position PV zero. This walkthrough uses only the discount/projection curves.
Follow [setup](../../README.md#install-and-run), then run every cell in order.
For dates, market construction and result units, start with [the swap tour](tour.ipynb).

In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date


## A swap: value the contract and solve its par coupon

The fixed rate is the term we vary. Everything else stays identical. Explicit
trade dates matter for forward-starting products: the default trade date on
many specifications is their start date, while we want valuation as of `ref`.

In [2]:
def receive_fixed_swap(fixed_rate):
    instrument = qs.Swap(
        identifier="USD-SWAP-5Y",
        start_date=ref,
        maturity_date=ref + "5Y",
        notional=1_000_000,
        fixed_rate=fixed_rate,
        currency=qs.Currency.USD,
        market_index=qs.MarketIndex.SOFR,
        fixed_leg_frequency=qs.Frequency.Semiannual,
        floating_leg_frequency=qs.Frequency.Quarterly,
    )
    return qs.Trade(instrument, ref, qs.Side.LongReceive)


context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
)

with context:
    swap_result = context.evaluate(
        receive_fixed_swap(0.04),
        [qs.Request.Value, qs.Request.FairRate],
    )
    par_result = context.evaluate(
        receive_fixed_swap(swap_result.fair_rate),
        [qs.Request.Value],
    )

pd.Series(
    {
        "original_pv_usd": swap_result.price,
        "par_rate": swap_result.fair_rate,
        "pv_at_par_usd": par_result.price,
    }
)

original_pv_usd   -1.664892e+03
par_rate           4.036800e-02
pv_at_par_usd     -2.728484e-11
dtype: float64

In [3]:
assert abs(par_result.price) < 1e-5
print("The solved par coupon reprices the swap to zero.")

The solved par coupon reprices the swap to zero.


Useful swap requests are `Value`, `FairRate`, `Cashflows` and `Sensitivities`.
`price` is position PV in USD; `fair_rate` is an annual decimal coupon.
Continue with [currency conversion](currency_conversion.ipynb) or
[portfolio risk](../../sensitivity/portfolio_risk.ipynb).